# Lab 03: Geometric Transformations
**Full 360° Coding Knowledge: Minimalist, Vectorized, Pure Matrix Algebra**


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(a, b, t1="Input", t2="Output"):
    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(a, cv2.COLOR_BGR2RGB)); plt.title(t1); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(b, cv2.COLOR_BGR2RGB)); plt.title(t2); plt.axis("off")
    plt.tight_layout(); plt.show()


### Task 1: Linear Scale (Enlarge 300% & Center)


In [ ]:
def task1(img):
    h, w = img.shape[:2]
    s = 3.0
    S = np.diag([s, s])
    t = np.float32([[(w / 2) * (1 - s)], [(h / 2) * (1 - s)]])
    M = np.hstack([S, t])
    return cv2.warpAffine(img, M, (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.circle(img, (180, 180), 50, (0, 255, 180), 2)
show(img, task1(img), "Tiny Fingerprint", "Enlarged 300% Centered")


### Task 2: Linear Rotation (North & Non-Clipping Bounds)


In [ ]:
def task2(img):
    h, w = img.shape[:2]
    th = np.radians(-45)
    c, s = abs(np.cos(th)), abs(np.sin(th))
    nw, nh = int(w * c + h * s), int(h * c + w * s)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), -45, 1.0)
    M[:, 2] += [(nw - w) / 2, (nh - h) / 2]
    return cv2.warpAffine(img, M, (nw, nh))

img = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.rectangle(img, (100, 100), (260, 260), (255, 160, 50), -1)
show(img, task2(img), "Rotated Satellite", "True North (Unclipped)")


### Task 3: Linear Shear (Horizontal Slant Correction)


In [ ]:
def task3(img):
    h, w = img.shape[:2]
    sh = 0.35
    S = np.float32([[1, sh], [0, 1]])
    t = np.float32([[-sh * w / 2], [0]])
    M = np.hstack([S, t])
    return cv2.warpAffine(img, M, (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
for x in range(60, 300, 20):
    cv2.line(img, (x + 30, 100), (x - 30, 260), (255, 255, 255), 4)
show(img, task3(img), "Slanted Barcode", "Straight Barcode")


### Task 4: Affine Translation (Slide Hidden Map)


In [ ]:
def task4(img):
    h, w = img.shape[:2]
    T = np.float32([[1, 0, 150], [0, 1, 80], [0, 0, 1]])
    return cv2.warpAffine(img, T[:2], (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.drawMarker(img, (30, 20), (0, 0, 255), cv2.MARKER_TILTED_CROSS, 25, 3)
show(img, task4(img), "Offscreen Map", "Shifted to View")


### Task 5: Rigid Transformation (Robot Gripper)


In [ ]:
def task5(img):
    h, w = img.shape[:2]
    th = np.radians(30)
    c, s = np.cos(th), np.sin(th)
    R = np.float32([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    T = np.float32([[1, 0, 40], [0, 1, 80], [0, 0, 1]])
    M = (T @ R)[:2]
    return cv2.warpAffine(img, M, (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.rectangle(img, (140, 100), (220, 160), (0, 220, 220), -1)
show(img, task5(img), "Misaligned Chip", "Gripper Aligned")


### Task 6: Similarity Transformation (Architect Blueprint)


In [ ]:
def task6(img):
    h, w = img.shape[:2]
    s = 1.8
    th = np.radians(-35)
    c, sn = np.cos(th), np.sin(th)
    S = np.diag([s, s, 1.0])
    R = np.float32([[c, -sn, 0], [sn, c, 0], [0, 0, 1]])
    T = np.float32([[1, 0, 100], [0, 1, 90], [0, 0, 1]])
    M = (T @ R @ S)[:2]
    return cv2.warpAffine(img, M, (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.rectangle(img, (40, 40), (120, 120), (255, 100, 180), 2)
show(img, task6(img), "Wrong Blueprint", "Similarity Fixed")


### Task 7: General Affine (Solve Linear System for 6 Unknowns)


In [ ]:
def task7(img):
    h, w = img.shape[:2]
    p1 = np.float32([[50, 50], [200, 50], [100, 200]])
    p2 = np.float32([[70, 100], [220, 40], [150, 240]])
    A = np.c_[p1, np.ones(3)]
    u = np.linalg.solve(A, p2[:, 0])
    v = np.linalg.solve(A, p2[:, 1])
    M = np.float32([u, v])
    return cv2.warpAffine(img, M, (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.polylines(img, [np.int32([[60, 60], [300, 90], [180, 290]])], True, (0, 255, 255), 2)
show(img, task7(img), "Glitched Photo", "Affine Solved")


### Task 8: Perspective (Bird's-Eye Square)


In [ ]:
def task8(img):
    h, w = img.shape[:2]
    src = np.float32([[120, 80], [240, 80], [310, 310], [50, 310]])
    dst = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    H = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, H, (w, h))

img = np.zeros((360, 360, 3), dtype=np.uint8)
pts = np.int32([[120, 80], [240, 80], [310, 310], [50, 310]])
cv2.fillPoly(img, [pts], (180, 100, 255))
show(img, task8(img), "Chalk Perspective", "Bird's-Eye Square")


### Task 9: Homography Panorama


In [ ]:
def task9(cam1, cam2):
    h, w = cam1.shape[:2]
    p2 = np.float32([[60, 60], [160, 60], [160, 280], [60, 280]])
    p1 = np.float32([[200, 70], [300, 50], [310, 290], [210, 270]])
    H, _ = cv2.findHomography(p2, p1)
    pano = cv2.warpPerspective(cam2, H, (w * 2, h))
    pano[:, :w] = np.where(cam1 > 0, cam1, pano[:, :w])
    return pano

cam1 = np.zeros((360, 360, 3), dtype=np.uint8)
cam2 = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.rectangle(cam1, (50, 50), (310, 310), (0, 140, 0), -1)
cv2.rectangle(cam2, (50, 50), (310, 310), (0, 180, 0), -1)

res = task9(cam1, cam2)
plt.figure(figsize=(7, 3))
plt.imshow(cv2.cvtColor(res, cv2.COLOR_BGR2RGB))
plt.title("Stadium Panorama Stitch")
plt.axis("off"); plt.show()


### Task 10: Full Hierarchy (Scale -> Rigid -> Projective)


In [ ]:
def task10(wall, paint):
    h, w = paint.shape[:2]
    th = np.radians(15)
    c, s = np.cos(th), np.sin(th)
    S = np.diag([0.5, 0.5, 1.0])
    R = np.float32([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    T = np.float32([[1, 0, 80], [0, 1, 60], [0, 0, 1]])
    step12 = cv2.warpAffine(paint, (T @ R @ S)[:2], (w, h))
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([[100, 60], [280, 90], [250, 310], [80, 270]])
    H = cv2.getPerspectiveTransform(src, dst)
    res = cv2.warpPerspective(step12, H, (w, h))
    wall[res > 0] = res[res > 0]
    return wall

wall = np.full((360, 360, 3), 40, dtype=np.uint8)
paint = np.zeros((360, 360, 3), dtype=np.uint8)
cv2.circle(paint, (180, 180), 90, (50, 200, 255), -1)
show(paint, task10(wall, paint), "Painting", "Framed in Angled Museum Wall")
